In [ ]:
"""
Descarga automática de establecimientos MINEDUC (modo headless).
- Descarga todos los departamentos y niveles BASICO/DIVERSIFICADO.
- Unifica en un solo CSV.
- Elimina los archivos .xls descargados al final.
"""

import os
import time
import glob
import shutil
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException

# ============================================================================
# CONFIGURACIÓN
# ============================================================================
BASE_URL = "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/"

DEPARTAMENTOS = [
    "ALTA VERAPAZ", "BAJA VERAPAZ", "CHIMALTENANGO", "CHIQUIMULA",
    "CIUDAD CAPITAL", "EL PROGRESO", "ESCUINTLA", "GUATEMALA",
    "HUEHUETENANGO", "IZABAL", "JALAPA", "JUTIAPA", "PETEN",
    "QUETZALTENANGO", "QUICHE", "RETALHULEU", "SACATEPEQUEZ",
    "SAN MARCOS", "SANTA ROSA", "SOLOLA", "SUCHITEPEQUEZ",
    "TOTONICAPAN", "ZACAPA",
]

# Solo niveles de interés (Básico y Diversificado)
NIVELES = ["BASICO", "DIVERSIFICADO"]

# Carpeta de descargas (dentro del directorio actual)
CARPETA_DESCARGAS = os.path.join(os.getcwd(), "descargas")
os.makedirs(CARPETA_DESCARGAS, exist_ok=True)

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def encontrar_select_por_nombre(driver, nombre_parcial):
    selects = driver.find_elements(By.TAG_NAME, "select")
    for sel in selects:
        name = sel.get_attribute("name")
        if name and nombre_parcial in name:
            return sel
    return None

def encontrar_boton_por_src(driver, src_parcial):
    inputs = driver.find_elements(By.XPATH, "//input[@type='image']")
    for inp in inputs:
        src = inp.get_attribute("src")
        if src and src_parcial in src:
            return inp
    return None

def hacer_scroll_hasta_elemento(driver, elemento):
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elemento)
    time.sleep(0.5)

def clic_seguro(driver, elemento):
    try:
        elemento.click()
    except:
        from selenium.webdriver.common.action_chains import ActionChains
        ActionChains(driver).move_to_element(elemento).click().perform()

def esperar_descarga(carpeta, timeout=30):
    """Espera a que aparezca un nuevo archivo .xls en la carpeta."""
    archivos_iniciales = set(glob.glob(os.path.join(carpeta, "*.xls")))
    start = time.time()
    while time.time() - start < timeout:
        archivos_actuales = set(glob.glob(os.path.join(carpeta, "*.xls")))
        nuevos = archivos_actuales - archivos_iniciales
        if nuevos:
            return max(nuevos, key=os.path.getctime)
        time.sleep(1)
    raise TimeoutError("No se detectó la descarga del archivo .xls")

def leer_tabla_desde_archivo_xls(ruta_archivo):
    """
    Lee el archivo .xls (que en realidad es HTML) y extrae la tabla de datos.
    Devuelve un DataFrame con los datos limpios.
    """
    with open(ruta_archivo, 'r', encoding='latin1') as f:
        contenido = f.read()
    tablas = pd.read_html(contenido)
    if not tablas:
        raise ValueError("No se encontraron tablas en el archivo")
    df = max(tablas, key=lambda t: t.shape[0])
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    df = df.dropna(how='all')
    return df

# ============================================================================
# FUNCIÓN PRINCIPAL: descarga un departamento + nivel
# ============================================================================

def descargar_departamento_nivel(driver, departamento, nivel):
    driver.get(BASE_URL)
    wait = WebDriverWait(driver, 15)

    # 1. Departamento
    select_depto = encontrar_select_por_nombre(driver, "cmbDepartamento")
    if not select_depto:
        raise Exception("No se encontró el select de Departamento")
    Select(select_depto).select_by_visible_text(departamento)
    time.sleep(1.5)

    # 2. Nivel
    select_nivel = encontrar_select_por_nombre(driver, "cmbNivel")
    if not select_nivel:
        raise Exception("No se encontró el select de Nivel")
    Select(select_nivel).select_by_visible_text(nivel)
    time.sleep(0.5)

    # 3. Buscar
    boton_buscar = driver.find_element(By.XPATH, "//input[@type='image' and contains(@src, 'seleccionar_estab.gif')]")
    hacer_scroll_hasta_elemento(driver, boton_buscar)
    clic_seguro(driver, boton_buscar)
    time.sleep(4)

    # 4. Esperar tabla de resultados
    try:
        wait.until(EC.presence_of_element_located((By.XPATH, "//table")))
    except TimeoutException:
        print(f"  Sin resultados para {departamento} - {nivel}")
        return None

    # 5. Botón Excel
    boton_excel = encontrar_boton_por_src(driver, "export_excel.gif")
    if not boton_excel:
        try:
            boton_excel = driver.find_element(By.XPATH, "//*[contains(text(),'Generar Archivo de Excel')]")
        except:
            pass
    if not boton_excel:
        raise Exception("No se encontró el botón de Excel")

    hacer_scroll_hasta_elemento(driver, boton_excel)
    wait.until(EC.element_to_be_clickable(boton_excel))
    boton_excel.click()

    # 6. Esperar descarga
    ruta_archivo = esperar_descarga(CARPETA_DESCARGAS, timeout=30)
    nuevo_nombre = f"{departamento}_{nivel}.xls"
    nueva_ruta = os.path.join(CARPETA_DESCARGAS, nuevo_nombre)
    if os.path.exists(nueva_ruta):
        base, ext = os.path.splitext(nuevo_nombre)
        nueva_ruta = os.path.join(CARPETA_DESCARGAS, f"{base}_{int(time.time())}{ext}")
    os.rename(ruta_archivo, nueva_ruta)
    print(f"  Descargado: {os.path.basename(nueva_ruta)}")
    return nueva_ruta

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    # Configurar Chrome en modo headless (sin ventana)
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    prefs = {
        "download.default_directory": CARPETA_DESCARGAS,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)

    try:
        archivos_descargados = []

        print("Iniciando descarga de todos los departamentos (Básico y Diversificado)...")
        for depto in DEPARTAMENTOS:
            for nivel in NIVELES:
                try:
                    print(f"Procesando {depto} - {nivel}...")
                    ruta = descargar_departamento_nivel(driver, depto, nivel)
                    if ruta:
                        archivos_descargados.append(ruta)
                except Exception as e:
                    print(f"  ERROR: {e}")
                time.sleep(2)  # pausa entre solicitudes

        print(f"\nArchivos descargados: {len(archivos_descargados)}")

        if not archivos_descargados:
            raise SystemExit("No se descargó ningún archivo.")

        # ---- Leer y unificar todos los archivos ----
        dataframes = []
        for archivo in archivos_descargados:
            try:
                df = leer_tabla_desde_archivo_xls(archivo)
                nombre_base = os.path.basename(archivo).replace(".xls", "")
                partes = nombre_base.split("_")
                if len(partes) >= 2:
                    df["DEPARTAMENTO_CONSULTADO"] = partes[0]
                    df["NIVEL_CONSULTADO"] = partes[1]
                dataframes.append(df)
                print(f"Leído: {os.path.basename(archivo)} -> {len(df)} filas")
            except Exception as e:
                print(f"Error al leer {archivo}: {e}")

        if not dataframes:
            raise SystemExit("No se pudieron leer los archivos.")

        df_unificado = pd.concat(dataframes, ignore_index=True)
        print(f"\nTotal filas (antes de deduplicar): {len(df_unificado)}")

        # ---- Limpieza ----
        df_unificado.columns = [str(c).strip().upper() for c in df_unificado.columns]

        if "CODIGO" in df_unificado.columns:
            df_unificado = df_unificado[df_unificado["CODIGO"].notna()]
            df_unificado = df_unificado[df_unificado["CODIGO"].astype(str).str.strip() != ""]

        categoricas = ["DEPARTAMENTO", "MUNICIPIO", "NIVEL", "SECTOR", "AREA",
                       "STATUS", "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL"]
        for col in categoricas:
            if col in df_unificado.columns:
                df_unificado[col] = df_unificado[col].astype(str).str.upper().str.strip()
                df_unificado[col] = df_unificado[col].replace({"NAN": pd.NA, "NONE": pd.NA, "---": pd.NA})

        if "TELEFONO" in df_unificado.columns:
            df_unificado["TELEFONO"] = df_unificado["TELEFONO"].astype(str).str.replace(r"\.0$", "", regex=True)
            df_unificado["TELEFONO"] = df_unificado["TELEFONO"].replace({"nan": pd.NA, "<NA>": pd.NA})

        if "CODIGO" in df_unificado.columns:
            antes = len(df_unificado)
            df_unificado["_no_nulos"] = df_unificado.notna().sum(axis=1)
            df_unificado = df_unificado.sort_values("_no_nulos", ascending=False).drop_duplicates(subset=["CODIGO"]).drop(columns="_no_nulos")
            print(f"Duplicados eliminados por CODIGO: {antes - len(df_unificado)}")

        print(f"Total filas final: {len(df_unificado)}")

        # ---- Guardar CSV final ----
        SALIDA_CSV = "mineduc_establecimientos_unificado.csv"
        df_unificado.to_csv(SALIDA_CSV, index=False, encoding="utf-8-sig")
        print(f"\nCSV guardado: {SALIDA_CSV} ({len(df_unificado)} filas, {len(df_unificado.columns)} columnas)")

        # ---- Eliminar archivos .xls descargados ----
        print("\nEliminando archivos .xls descargados para ahorrar espacio...")
        for archivo in archivos_descargados:
            try:
                os.remove(archivo)
                print(f"  Eliminado: {os.path.basename(archivo)}")
            except Exception as e:
                print(f"  No se pudo eliminar {archivo}: {e}")
        # Opcional: eliminar también la carpeta si queda vacía
        try:
            os.rmdir(CARPETA_DESCARGAS)
            print("  Carpeta 'descargas' eliminada.")
        except OSError:
            # La carpeta no está vacía (puede haber otros archivos)
            pass

        print("\nProceso completado exitosamente.")

    finally:
        driver.quit()

=== Prueba con ALTA VERAPAZ / BASICO ===
  Descargado: ALTA VERAPAZ_BASICO_1784149829.xls


C:\Users\olivi\AppData\Local\Temp\ipykernel_4468\967547530.py:92: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tablas = pd.read_html(contenido)


Filas obtenidas: 761
0         CODIGO    DISTRITO  DEPARTAMENTO MUNICIPIO  \
0  16-01-0026-45  16-01-0923  ALTA VERAPAZ     COBAN   
1  16-01-0135-45  16-01-0925  ALTA VERAPAZ     COBAN   
2  16-01-0136-45  16-01-0923  ALTA VERAPAZ     COBAN   
3  16-01-0137-45      16-006  ALTA VERAPAZ     COBAN   
4  16-01-0138-45  16-01-0926  ALTA VERAPAZ     COBAN   

0                                   ESTABLECIMIENTO  \
0                 COLEGIO PARTICULAR MIXTO IMPERIAL   
1  INEB ADSCRITO A INSTITUTO 'EMILIO ROSALES PONCE'   
2                                              INEB   
3      INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN   
4                                     COLEGIO COBAN   

0                                DIRECCION  TELEFONO  \
0                    5A. CALLE 1-98 ZONA 3  57101061   
1                      3A AVE 6-23 ZONA 11  54457786   
2                       6A AVE 1-15 ZONA 4  55541910   
3                       6A AVE 1-15 ZONA 4       NaN   
4  KM.2 SALIDA A SAN JUAN CHAME

C:\Users\olivi\AppData\Local\Temp\ipykernel_4468\967547530.py:92: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tablas = pd.read_html(contenido)


Leído: ALTA VERAPAZ_BASICO_1784149839.xls -> 761 filas


C:\Users\olivi\AppData\Local\Temp\ipykernel_4468\967547530.py:92: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tablas = pd.read_html(contenido)


Leído: ALTA VERAPAZ_DIVERSIFICADO.xls -> 475 filas
Leído: BAJA VERAPAZ_BASICO.xls -> 313 filas


C:\Users\olivi\AppData\Local\Temp\ipykernel_4468\967547530.py:92: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tablas = pd.read_html(contenido)
C:\Users\olivi\AppData\Local\Temp\ipykernel_4468\967547530.py:92: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tablas = pd.read_html(contenido)


Leído: BAJA VERAPAZ_DIVERSIFICADO.xls -> 171 filas

Total filas (antes de deduplicar): 1720
Duplicados eliminados por CODIGO: 0
Total filas final: 1720

CSV guardado: mineduc_establecimientos_unificado.csv (1720 filas, 19 columnas)
